# 03 — Pull V-Dem + Polity5

**Sources:**
- **V-Dem v15** (Varieties of Democracy) — 400+ annual governance indicators, 1789–present  
  Provider: V-Dem Institute, University of Gothenburg  
  Access: `vdemdata` Python package (bundles the dataset)  
- **Polity5** — regime type and executive constraints, 1800–2018  
  Provider: Center for Systemic Peace  
  Access: direct CSV download  

## What this notebook does
1. Loads the V-Dem panel via `vdemdata` and selects the ~20 indicators used across the literature
2. Downloads Polity5 from the Center for Systemic Peace and selects key regime variables
3. Writes each as a country-year parquet to ADLS

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

## Key variables
See DatasetVariableSynthesis.md §1.2 (V-Dem, Polity V) and meta-plan Domain 2.

In [ ]:
import os
import io
import requests
import pandas as pd
import vdemdata
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# Polity5 annual CSV — Center for Systemic Peace
POLITY5_URL = (
    "https://www.systemicpeace.org/inscr/p5v2018.csv"
)

print(f"Run date    : {RUN_DATE}")
print(f"ADLS target : abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/raw/")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## V-Dem: select indicators

The `vdemdata` package ships the full dataset; we select the subset relevant to instability
prediction from DatasetVariableSynthesis.md.

In [ ]:
# Core identifiers always kept
VDEM_ID_COLS = ["country_name", "country_text_id", "country_id", "year", "COWcode"]

# Regime-level democracy indices
VDEM_REGIME_COLS = [
    "v2x_libdem",          # Liberal Democracy Index
    "v2x_polyarchy",       # Electoral Democracy Index
    "v2x_partipdem",       # Participatory Democracy Index
    "v2x_egaldem",         # Egalitarian Democracy Index
    "v2x_delibdem",        # Deliberative Democracy Index
    "v2x_regime",          # Regime type (0–3 ordinal)
    "v2x_civlib",          # Civil Liberties Index
    "v2x_freexp_altinf",   # Freedom of Expression
]

# Early warning / backsliding indicators
VDEM_EWI_COLS = [
    "v2x_jucon",           # Judicial Constraints on Executive
    "v2x_legcon",          # Legislative Constraints on Executive
    "v2juncind",           # Judicial Independence
    "v2mecenefm",          # Media censorship effort
    "v2meharjrn",          # Harassment of journalists
    "v2cscnsult",          # CSO consultation
    "v2csreprss",          # CSO repression
    "v2elvotbuy",          # Vote buying
    "v2elfrfair",          # Free and fair elections
    "v2xcs_ccsi",          # Core Civil Society Index
    "v2x_corr",            # Political Corruption Index
    "v2regendtype",        # How regime ended (label use case)
    "v2exrescon",          # Executive respects constitution
]

VDEM_COLS = VDEM_ID_COLS + VDEM_REGIME_COLS + VDEM_EWI_COLS
print(f"Selecting {len(VDEM_COLS)} V-Dem columns ({len(VDEM_REGIME_COLS)} regime, {len(VDEM_EWI_COLS)} EWI)")

## Load V-Dem via vdemdata

In [ ]:
print("Loading V-Dem panel (this may take ~30 s for the full dataset)...")
df_vdem_full = vdemdata.load_data()
print(f"Full V-Dem shape: {df_vdem_full.shape}")

# Keep only columns that exist in this release
available = [c for c in VDEM_COLS if c in df_vdem_full.columns]
missing   = [c for c in VDEM_COLS if c not in df_vdem_full.columns]
if missing:
    print(f"  Note: {len(missing)} requested column(s) not in this release: {missing}")

df_vdem = df_vdem_full[available].copy()
df_vdem["year"] = df_vdem["year"].astype(int)

print(f"Filtered V-Dem shape: {df_vdem.shape}")
print(f"Countries: {df_vdem['country_name'].nunique()} | Years: {df_vdem['year'].min()}–{df_vdem['year'].max()}")
df_vdem.head(3)

## Write V-Dem to ADLS

In [ ]:
write_parquet(df_vdem, f"raw/vdem/{RUN_DATE}/vdem_panel.parquet")

## Polity5: select indicators

Polity5 is downloaded from the Center for Systemic Peace. The key variables are the
regime type composite (`polity2`) and the sub-components used to derive the
PITF 5-category regime classification — the single most powerful predictor in Goldstone (2010).

In [ ]:
POLITY5_COLS = {
    # Identifiers
    "ccode":   "cow_code",
    "country": "country_name",
    "year":    "year",
    # Core score
    "polity2": "polity2",      # −10 (autocracy) to +10 (democracy)
    "polity":  "polity_raw",   # raw score (may include special codes)
    "democ":   "democ",        # Institutionalised Democracy (0–10)
    "autoc":   "autoc",        # Institutionalised Autocracy (0–10)
    # PITF sub-components (for 5-category regime type)
    "xrreg":   "xrreg",        # Executive Recruitment Regulation
    "xrcomp":  "xrcomp",       # Executive Recruitment Competitiveness
    "xropen":  "xropen",       # Executive Recruitment Openness
    "xconst":  "xconst",       # Executive Constraints
    "parcomp": "parcomp",      # Competitiveness of Participation
    "parreg":  "parreg",       # Regulation of Participation
    # Derived
    "durable": "regime_durability",  # years since last regime change
    "fragment":"state_fragment",     # state fragmentation flag
    "regtrans":"regime_transition",  # transition code
}

## Download and parse Polity5

In [ ]:
print(f"Downloading Polity5 from {POLITY5_URL} ...")
resp = requests.get(POLITY5_URL, timeout=60)
resp.raise_for_status()

df_polity_raw = pd.read_csv(io.BytesIO(resp.content), low_memory=False)
print(f"Raw shape: {df_polity_raw.shape}")
print(f"Columns: {list(df_polity_raw.columns[:10])} ...")

In [ ]:
# Keep only columns we requested that exist in the file
available_p5 = {k: v for k, v in POLITY5_COLS.items() if k in df_polity_raw.columns}
missing_p5   = [k for k in POLITY5_COLS if k not in df_polity_raw.columns]
if missing_p5:
    print(f"  Note: columns not found in Polity5 file: {missing_p5}")

df_polity = df_polity_raw[list(available_p5.keys())].rename(columns=available_p5).copy()
df_polity["year"] = df_polity["year"].astype(int)

# Replace Polity special codes (−66=interruption, −77=interregnum, −88=transition) with NaN
for col in ["polity2", "polity_raw", "democ", "autoc"]:
    if col in df_polity.columns:
        df_polity[col] = df_polity[col].replace([-66, -77, -88], pd.NA)
        df_polity[col] = pd.to_numeric(df_polity[col], errors="coerce")

# Derive PITF anocracy flag: polity2 in [−5, +5]
if "polity2" in df_polity.columns:
    df_polity["anocracy_flag"] = (
        df_polity["polity2"].between(-5, 5, inclusive="both")
    ).astype("Int8")

print(f"Polity5 shape: {df_polity.shape}")
print(f"Countries: {df_polity['cow_code'].nunique() if 'cow_code' in df_polity.columns else 'n/a'} | Years: {df_polity['year'].min()}–{df_polity['year'].max()}")
df_polity.head(3)

## Write Polity5 to ADLS

In [ ]:
write_parquet(df_polity, f"raw/polity5/{RUN_DATE}/polity5_panel.parquet")

## Summary

In [ ]:
print("=" * 55)
print("V-Dem + Polity5 pull complete")
print("=" * 55)
print(f"  V-Dem  : {len(df_vdem):,} country-years | {df_vdem['country_name'].nunique()} countries")
print(f"  Polity5: {len(df_polity):,} country-years")
print()
print("ADLS paths written:")
print(f"  raw/vdem/{RUN_DATE}/vdem_panel.parquet")
print(f"  raw/polity5/{RUN_DATE}/polity5_panel.parquet")